[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/llms/06-finetuning-lora.ipynb)

# 06 — Fine-tuning con LoRA

> **Bloque:** LLMs | **Nivel:** Avanzado
>
> Complementario al tutorial [06-finetuning-lora.md](../../llms/06-finetuning-lora.md)

Este notebook cubre:
1. Generar un dataset de entrenamiento con Claude
2. Preparar y validar el dataset
3. Configuración QLoRA (ejecutar en GPU)
4. Fine-tuning gestionado con la API de OpenAI (sin GPU)

> **Nota:** Las celdas de entrenamiento local requieren GPU con ≥8 GB VRAM. Para ejecutarlas sin GPU, usa Google Colab con GPU T4.

In [ ]:
# %pip install anthropic openai python-dotenv datasets pandas matplotlib

import anthropic
import json
import random
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline

client = anthropic.Anthropic()
print('✓ Listo')

## 1. Generar dataset sintético con Claude

Creamos un dataset de clasificación de tickets de soporte. Claude genera ejemplos variados y realistas.

In [ ]:
CATEGORIAS = ['BUG', 'FEATURE_REQUEST', 'PREGUNTA', 'QUEJA', 'OTRO']

def generar_ejemplos(categoria: str, n: int = 15) -> list:
    prompt = f"""Genera {n} ejemplos realistas de tickets de soporte de la categoría '{categoria}'.
Cada ejemplo debe ser diferente. Varía la longitud, tono y formulación.
Devuelve SOLO un JSON array: [{{"texto": "..."}}]"""

    r = client.messages.create(
        model='claude-haiku-4-5-20251001', max_tokens=1500, temperature=0.85,
        messages=[{'role': 'user', 'content': prompt}]
    )
    raw = r.content[0].text.strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'): raw = raw[4:]
    ejemplos = json.loads(raw)
    return [{'instruction': 'Clasifica este ticket de soporte.',
             'input': e['texto'], 'output': categoria} for e in ejemplos]

print('Generando dataset sintético...')
dataset_completo = []
for cat in CATEGORIAS:
    print(f'  Generando {cat}...', end=' ')
    ejemplos = generar_ejemplos(cat, n=10)
    dataset_completo.extend(ejemplos)
    print(f'✓ {len(ejemplos)} ejemplos')

random.shuffle(dataset_completo)
print(f'\nTotal: {len(dataset_completo)} ejemplos')

## 2. Explorar y validar el dataset

In [ ]:
df = pd.DataFrame(dataset_completo)
df['longitud'] = df['input'].apply(lambda x: len(x.split()))

print('=== Distribución por categoría ===')
print(df['output'].value_counts())

print(f'\n=== Longitud de textos ===')
print(df['longitud'].describe().round(1))

# Visualización
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

conteo = df['output'].value_counts()
ax1.bar(conteo.index, conteo.values, color=['#3498DB', '#2ECC71', '#E74C3C', '#9B59B6', '#F39C12'],
        alpha=0.85, edgecolor='black')
ax1.set_title('Ejemplos por categoría')
ax1.set_ylabel('Número de ejemplos')

df.boxplot(column='longitud', by='output', ax=ax2)
ax2.set_title('Longitud por categoría (palabras)')
ax2.set_xlabel('Categoría')
plt.suptitle('')

plt.tight_layout()
plt.show()

# Mostrar 3 ejemplos
print('\n=== Muestra del dataset ===')
df.sample(3)[['input', 'output']].to_string(index=False)

## 3. Guardar en formato JSONL

In [ ]:
# Dividir en train/validation
split = int(len(dataset_completo) * 0.8)
train = dataset_completo[:split]
val   = dataset_completo[split:]

def guardar_jsonl(datos: list, ruta: str):
    with open(ruta, 'w', encoding='utf-8') as f:
        for d in datos:
            f.write(json.dumps(d, ensure_ascii=False) + '\n')

Path('datos').mkdir(exist_ok=True)
guardar_jsonl(train, 'datos/tickets_train.jsonl')
guardar_jsonl(val,   'datos/tickets_val.jsonl')

print(f'Train: {len(train)} ejemplos → datos/tickets_train.jsonl')
print(f'Val:   {len(val)} ejemplos → datos/tickets_val.jsonl')

## 4. Fine-tuning gestionado con OpenAI (sin GPU)

La forma más sencilla de hacer fine-tuning sin infraestructura propia.

In [ ]:
# Convertir al formato que requiere OpenAI (ChatML)
def convertir_a_chatml(ejemplos: list) -> list:
    return [{
        'messages': [
            {'role': 'system', 'content': 'Clasifica tickets de soporte en: BUG, FEATURE_REQUEST, PREGUNTA, QUEJA u OTRO.'},
            {'role': 'user',   'content': ejemplo['input']},
            {'role': 'assistant', 'content': ejemplo['output']}
        ]
    } for ejemplo in ejemplos]

train_oai = convertir_a_chatml(train)
guardar_jsonl(train_oai, 'datos/tickets_train_openai.jsonl')
print(f'Dataset en formato OpenAI: {len(train_oai)} ejemplos')
print('\nEjemplo:')
print(json.dumps(train_oai[0], ensure_ascii=False, indent=2))

In [ ]:
# DESCOMENTA para ejecutar el fine-tuning con OpenAI
# Requiere OPENAI_API_KEY y saldo en la cuenta

# from openai import OpenAI
# client_oai = OpenAI()
#
# # 1. Subir dataset
# with open('datos/tickets_train_openai.jsonl', 'rb') as f:
#     fichero = client_oai.files.create(file=f, purpose='fine-tune')
# print(f'Fichero subido: {fichero.id}')
#
# # 2. Lanzar fine-tuning
# job = client_oai.fine_tuning.jobs.create(
#     training_file=fichero.id,
#     model='gpt-4o-mini-2024-07-18'
# )
# print(f'Job: {job.id} | Estado: {job.status}')
#
# # 3. Monitorizar (ejecutar varias veces hasta que status == 'succeeded')
# job_info = client_oai.fine_tuning.jobs.retrieve(job.id)
# print(f'Estado: {job_info.status}')
# if job_info.fine_tuned_model:
#     print(f'Modelo: {job_info.fine_tuned_model}')

print('Celda lista — descomenta el código para ejecutar el fine-tuning con OpenAI.')

## 5. Configuración QLoRA local (requiere GPU)

In [ ]:
# Esta celda muestra la configuración — ejecutar requiere GPU con ≥8GB VRAM
# En Google Colab: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4

config_qlora = {
    'modelo_base': 'meta-llama/Meta-Llama-3-8B-Instruct',
    'cuantizacion': {
        'load_in_4bit': True,
        'bnb_4bit_quant_type': 'nf4',
        'bnb_4bit_compute_dtype': 'bfloat16',
    },
    'lora': {
        'r': 16,
        'lora_alpha': 32,
        'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        'lora_dropout': 0.05,
    },
    'entrenamiento': {
        'num_train_epochs': 3,
        'per_device_train_batch_size': 4,
        'gradient_accumulation_steps': 4,
        'learning_rate': 2e-4,
        'bf16': True,
        'optim': 'paged_adamw_32bit',
    }
}

print('Configuración QLoRA:')
print(json.dumps(config_qlora, indent=2))

# Parámetros entrenables estimados
params_modelo = 8_000_000_000
r = config_qlora['lora']['r']
# 4 módulos de atención × 2 matrices LoRA × dim × r (estimado)
params_lora_estimado = 4 * 2 * 4096 * r
porcentaje = params_lora_estimado / params_modelo * 100
print(f'\nParámetros entrenables estimados: ~{params_lora_estimado/1e6:.1f}M ({porcentaje:.2f}% del modelo)')

## 6. Evaluar el baseline antes del fine-tuning

In [ ]:
# Evaluamos cuánto acierta Claude Haiku SIN fine-tuning
import time

def clasificar_baseline(texto: str) -> str:
    prompt = f"""Clasifica este ticket en: BUG, FEATURE_REQUEST, PREGUNTA, QUEJA u OTRO.
Responde SOLO con la categoría.

Ticket: "{texto}""""
    r = client.messages.create(
        model='claude-haiku-4-5-20251001', max_tokens=20, temperature=0.0,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return r.content[0].text.strip().upper()

# Evaluar en una muestra del set de validación
muestra = random.sample(val, min(20, len(val)))
correctos = 0

for ejemplo in muestra:
    predicho = clasificar_baseline(ejemplo['input'])
    if predicho == ejemplo['output']:
        correctos += 1
    time.sleep(0.2)

acc_baseline = correctos / len(muestra)
print(f'Accuracy baseline (sin fine-tuning): {acc_baseline:.1%} ({correctos}/{len(muestra)})')
print('\nEste es el punto de partida. El fine-tuning debería mejorarlo significativamente.')

---
**Tutorial relacionado:** [06-finetuning-lora.md](../../llms/06-finetuning-lora.md)